In [9]:
import os
import sys
import yaml
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.append(str('/scratch/users/linyao/ML4MJO/scripts/src'))

# ======================================================================
# 1. Environment & Path Setup
# ======================================================================

import utils.metrics as mjo

# Read environment variables
dataflg = os.environ.get("dataflg", "era5").lower()
input_var_name = os.environ.get("input_var", "olr")
output_var_name = os.environ.get("output_var", "RMM")
model_name = os.environ.get("model_name", "UNet_A")
multi_lead = os.environ.get("multi_lead", "true").lower() == "true"

# Define Evaluation Periods
PERIODS = {
    # "val": {"start": "2010-01-01", "end": "2015-12-31"},  # Uncomment if needed
    "test": {"start": "2016-01-01", "end": "2021-12-31"}
}

# ======================================================================
# 2. Bootstrap & Experiment Configuration
# ======================================================================
N_ENSEMBLES = 16
SUBSET_SIZE = 11
N_BOOTSTRAPS = 20  # Adjust as needed. 100 is a good balance for I/O time.
TOP_TRIALS = [1,]

# Define the groups of experiments
groups = {
    # "Rescaled_Group": [
    #     'fltano120', 
    #     'rescaled_m10all_wnx1off', 
    #     'rescaled_m10all_wnx9all', 
    #     'rescaled_m10resi_wnx9resi'
    # ],
    "Unscaled_Group": [
        'fltano120', 
        # 'unscaled_m10all_wnx1off', 
        'unscaled_m10all_wnx9all', 
        'unscaled_m10resi_wnx9resi'
    ]
}

COLORS = ['black', 'tab:blue', 'red']

BASE_PRED_PATH = "/scratch/users/linyao/ML4MJO/scripts/outputs/predictions/{dataset_type}/{model_name}/{target_name}/lead{lead}/{exp_num}/preds_lr{lr}_bs{batch_size}_do{dropout}_cnn{channels_list_str}_k{kernel_size}_mlp{hidden_layers_str}_{optimizer}.nc"
metrics_base_dir = Path("./")

print("\n" + "="*50)
print("=== PART 1: BOOTSTRAP METRIC CALCULATIONS ===")
print("="*50)

# ======================================================================
# 3. Calculation Loop
# ======================================================================
for group_name, expflg_list in groups.items():
    print(f"\nProcessing Group: {group_name}")
    
    for expflg in expflg_list:
        if not multi_lead:
            lead = int(os.environ.get("lead", 15))
            exp_name = f"{dataflg}_{input_var_name}_{model_name}_{output_var_name}_{expflg}_lead{lead}"
        else:
            exp_name = f"{dataflg}_{input_var_name}_{model_name}_{output_var_name}_{expflg}"
            
        save_dir = metrics_base_dir / exp_name
        save_dir.mkdir(parents=True, exist_ok=True)
        
        for trial_rank in TOP_TRIALS:
            trial_tag = f"t{trial_rank}"
            config_path = f'./yaml/best_config_{exp_name}_{trial_tag}.yaml'
            
            if not os.path.exists(config_path):
                print(f"[{expflg}] Config not found: {config_path}. Skipping.")
                continue
                
            with open(config_path, "r") as f:
                config = yaml.safe_load(f)

            lead_conf = config["data"]["lead"]
            target_path = config["data"]["target_path"]
            
            save_kwargs = {
                "dataset_type": exp_name,
                "lat_range": config["data"]["lat_range"],
                "lead": lead_conf,
                "lr": config["training"]["learning_rate"],
                "batch_size": config["training"]["batch_size"],
                "dropout": config["model"]["mlp"]["dropout"],
                "channels_list_str": config["model"]["cnn"]["channels_list_str"],
                "kernel_size": config.get("kernel_size_str", f"{config['model']['cnn']['kernel_size'][0]}_{config['model']['cnn']['kernel_size'][1]}"),
                "hidden_layers_str": config.get("hidden_layers_str", "_".join(map(str, config["model"]["mlp"]["hidden_layers"]))),
                "optimizer": config["training"]["optimizer"],
                "model_name": config["model"]["name"],
                "target_name": config["data"]["target_vars"][0]
            }

            # Build full list of 16 prediction files
            fn_list = []
            for exp_num in range(1, N_ENSEMBLES + 1):
                save_kwargs["exp_num"] = f"{trial_tag}/exp{exp_num}"
                fn = BASE_PRED_PATH.format(**save_kwargs)
                if os.path.exists(fn):
                    fn_list.append(fn)

            if len(fn_list) < SUBSET_SIZE:
                print(f"[{expflg}] Not enough prediction files to sample {SUBSET_SIZE}. Skipping.")
                continue

            for period_name, dates in PERIODS.items():
                out_file = save_dir / f"boot_metrics_{exp_name}_{trial_tag}_{period_name}.npz"
                
                if os.path.exists(out_file):
                    print(f"  -> Bootstrap metrics already exist for {expflg} ({period_name}).")
                    continue

                print(f"\n--- Bootstrapping {expflg} | {period_name.upper()} ---")
                
                boot_ens_bcc = []
                boot_ens_rmse = []

                for b in range(N_BOOTSTRAPS):
                    if (b + 1) % 10 == 0:
                        print(f"     Iteration {b + 1}/{N_BOOTSTRAPS}...")
                    
                    # Randomly sample 11 member filenames from the 16
                    sub_fn_list = random.sample(fn_list, SUBSET_SIZE)
                    
                    # Compute skill *using the subset ensemble mean*
                    if multi_lead:
                        ens_bcc, ens_rmse = mjo.get_skill_all_leads_ensemble_mean(
                            fn_list=sub_fn_list,
                            datesta=dates["start"],
                            dateend=dates["end"],
                            leadmjo=lead_conf,
                            Fnmjo=target_path
                        )
                    else:
                        ens_bcc, ens_rmse = mjo.get_skill_ensemble_mean(
                            fn_list=sub_fn_list,
                            datesta=dates["start"],
                            dateend=dates["end"],
                            leadmjo=lead_conf,
                            Fnmjo=target_path
                        )            
                    
                    boot_ens_bcc.append(ens_bcc)
                    boot_ens_rmse.append(ens_rmse)
                
                # Convert to numpy arrays and calculate mean/spread bounds across bootstraps
                boot_ens_bcc = np.array(boot_ens_bcc)
                boot_ens_rmse = np.array(boot_ens_rmse)
                
                np.savez(
                    out_file,
                    bcc_mean=np.mean(boot_ens_bcc, axis=0),
                    bcc_min=np.min(boot_ens_bcc, axis=0),
                    bcc_max=np.max(boot_ens_bcc, axis=0),
                    rmse_mean=np.mean(boot_ens_rmse, axis=0),
                    rmse_min=np.min(boot_ens_rmse, axis=0),
                    rmse_max=np.max(boot_ens_rmse, axis=0),
                    lead=lead_conf,
                    trial_rank=trial_rank
                )
                print(f"Saved {period_name.upper()} bootstrap metrics to: {out_file}")




=== PART 1: BOOTSTRAP METRIC CALCULATIONS ===

Processing Group: Unscaled_Group

--- Bootstrapping fltano120 | TEST ---
     Iteration 10/20...
     Iteration 20/20...
Saved TEST bootstrap metrics to: era5_olr_UNet_A_RMM_fltano120/boot_metrics_era5_olr_UNet_A_RMM_fltano120_t1_test.npz

--- Bootstrapping unscaled_m10all_wnx9all | TEST ---
     Iteration 10/20...
     Iteration 20/20...
Saved TEST bootstrap metrics to: era5_olr_UNet_A_RMM_unscaled_m10all_wnx9all/boot_metrics_era5_olr_UNet_A_RMM_unscaled_m10all_wnx9all_t1_test.npz

--- Bootstrapping unscaled_m10resi_wnx9resi | TEST ---
     Iteration 10/20...
     Iteration 20/20...
Saved TEST bootstrap metrics to: era5_olr_UNet_A_RMM_unscaled_m10resi_wnx9resi/boot_metrics_era5_olr_UNet_A_RMM_unscaled_m10resi_wnx9resi_t1_test.npz


In [17]:
dataflg = 'era5'
output_var_name = 'ROMI'

plot_save_dir = metrics_base_dir / "group_comparisons_bootstrapped"
plot_save_dir.mkdir(parents=True, exist_ok=True)

plt.rcParams['font.size'] = 20

# We only process the first trial tag based on TOP_TRIALS list
trial_tag = f"t{TOP_TRIALS[0]}"

for group_name, expflg_list in groups.items():
    print(f"\nPlotting Group: {group_name}")
    
    for period_name in PERIODS.keys():
        
        fig_bcc, ax_bcc = plt.subplots(1, 1, figsize=(8.5, 6.5))
        # fig_rmse, ax_rmse = plt.subplots(1, 1, figsize=(8.5, 6.5))
        
        for i, expflg in enumerate(expflg_list):
            if multi_lead:
                exp_name = f"{dataflg}_{input_var_name}_{model_name}_{output_var_name}_{expflg}"
            else:
                lead = int(os.environ.get("lead", 15))
                exp_name = f"{dataflg}_{input_var_name}_{model_name}_{output_var_name}_{expflg}_lead{lead}"
                
            data_path = metrics_base_dir / exp_name / f"boot_metrics_{exp_name}_{trial_tag}_{period_name}.npz"
            
            if not os.path.exists(data_path):
                print(f"  -> Missing data for {expflg}. Skipping plot line.")
                continue
                
            data = np.load(data_path)
            lead_max = int(data['lead']) if multi_lead else lead
            leads = np.arange(0, lead_max + 1) if multi_lead else [lead]
            
            # Load the aggregated bootstrap metrics
            bcc_mean, bcc_min, bcc_max = data['bcc_mean'], data['bcc_min'], data['bcc_max']
            # rmse_mean, rmse_min, rmse_max = data['rmse_mean'], data['rmse_min'], data['rmse_max']
            
            # Plot BCC
            ax_bcc.fill_between(leads, bcc_min, bcc_max, color=COLORS[i], alpha=0.15)
            ax_bcc.plot(leads, bcc_mean, '-o', color=COLORS[i], linewidth=2, label=expflg)
            
            # # Plot RMSE
            # ax_rmse.fill_between(leads, rmse_min, rmse_max, color=COLORS[i], alpha=0.15)
            # ax_rmse.plot(leads, rmse_mean, '-o', color=COLORS[i], linewidth=2, label=expflg)

        # --- Formatting BCC Plot ---
        ax_bcc.axhline(y=0.5, color='black', linestyle='--', linewidth=2)
        ax_bcc.set_xticks(np.arange(0, 41, 5))
        ax_bcc.set_xlim(0, 35)
        ax_bcc.set_ylim(0.3, 0.9)
        ax_bcc.set_yticks(np.arange(0.3, 1.1, 0.2))
        ax_bcc.set_xlabel('Forecast lead (days)')
        ax_bcc.set_ylabel('BCC')
        ax_bcc.grid(True, linestyle=':', alpha=0.6)

        fig_bcc.tight_layout()
        bcc_save_path = plot_save_dir / f"plot_boot_bcc_{group_name}_{period_name}.png"
        fig_bcc.savefig(bcc_save_path, dpi=300)

        # # --- Formatting RMSE Plot ---
        # ax_rmse.axhline(y=1.2, color='black', linestyle='--', linewidth=2)
        # ax_rmse.axhline(y=np.sqrt(2), color='gray', linestyle='-.', linewidth=2)
        # ax_rmse.set_xticks(np.arange(0, 41, 5))
        # ax_rmse.set_xlim(0, 35)
        # ax_rmse.set_ylim(0, 1.8)
        # ax_rmse.set_xlabel('Forecast lead (days)')
        # ax_rmse.set_ylabel('RMSE')
        # ax_rmse.grid(True, linestyle=':', alpha=0.6)

        # fig_rmse.tight_layout()
        # rmse_save_path = plot_save_dir / f"plot_boot_rmse_{group_name}_{period_name}.png"
        # fig_rmse.savefig(rmse_save_path, dpi=300)
        
        plt.close(fig_bcc)
        # plt.close(fig_rmse)

print(f"\nAll plots generated and saved successfully to:\n{plot_save_dir}")


Plotting Group: Unscaled_Group

All plots generated and saved successfully to:
group_comparisons_bootstrapped
